In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import time
import random

In [3]:
df=pd.read_csv('../Multiple linear regression/Student_Performance.csv')

In [4]:
df.drop(columns=['Extracurricular Activities'],inplace=True)

In [5]:
df.sample(5)

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
8809,3,60,5,1,40.0
8721,2,94,5,0,64.0
2495,3,89,5,8,68.0
6751,7,75,6,6,66.0
2172,5,41,9,6,29.0


In [6]:
trf=ColumnTransformer([('trf_columns',StandardScaler(),[0,1,2,3])],remainder='passthrough')

In [7]:
df1=trf.fit_transform(df)


In [10]:
X=df1[:,0:4]
y=df1[:,-1]


In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [14]:
lr=LinearRegression()
lr.fit(X_train,y_train)

LinearRegression()

In [15]:
y_pred=lr.predict(X_test)
y_pred

array([55.00786377, 22.31444624, 47.59404705, ..., 16.48182916,
       63.64243731, 46.23799491])

In [16]:
coef=lr.coef_
intercept=lr.intercept_
print(f'intercept= {intercept}\ncoef={coef}')

intercept= 55.24069362844319
coef=[ 7.3866529  17.6377233   0.80264289  0.54971847]


In [18]:
from sklearn.metrics import r2_score

In [19]:
r2=r2_score(y_test,y_pred)
r2

0.9887144552384186

Mini Batch GD

In [29]:
class MGDRegressor:
    def __init__ (self,batch_size=10,learning_rate=0.01,epochs=100):
        self.coef_=None
        self.intercept_=None
        self.lr=learning_rate
        self.epoch=epochs
        self.batch_size=10
        
    def fit(self,X_train,y_train):
        self.coef_=np.ones(X_train.shape[1])
        self.intercept_=0
        for i in range(self.epoch):
            for j in range(int(X_train.shape[0]/self.batch_size)):
                idx=random.sample(range(0,X_train.shape[0]-1),10)
                y_pred=  X_train[idx] @ self.coef_ + self.intercept_  #X_train=(n,m),coef_=(m,1)
                slope_intercept=-2*(np.mean(y_train[idx] - y_pred))
                slope_coef = -2*((y_train[idx]-y_pred)@ X_train[idx])/X_train.shape[0]
                
                self.intercept_ = self.intercept_ - (self.lr * slope_intercept)
                self.coef_ = self.coef_ - (self.lr * slope_coef)
        print(self.intercept_,self.coef_)
        
    def predict(self,X_test):
        return X_test @ self.coef_ + self.intercept_

In [57]:
mbgd=MGDRegressor(batch_size=50,learning_rate=0.06,epochs=25)

In [58]:
mbgd.fit(X_train,y_train)

55.01810685895552 [ 7.04097471 16.80041918  0.82413841  0.60460167]


In [59]:
y_pred1=mbgd.predict(X_test)
y_pred1

array([54.77502515, 23.65674529, 47.76659403, ..., 18.0369921 ,
       63.04543049, 46.47796937])

In [60]:
r21=r2_score(y_test,y_pred1)
r21

0.9858993607701975